# Gabarito dos Exercícios Práticos - Processamento de Linguagem Natural (PLN)
**Fatec Jahu**

Este notebook contém as resoluções esperadas para os exercícios propostos, abordando desde a manipulação de áudio até a integração com modelos avançados de IA e otimização de hardware.

### Exercício 1: Parametrização da Gravação
Modifique a função `gravar_audio` para que o usuário possa definir a taxa de amostragem (`fs`), o número de canais e o tempo de gravação dinamicamente no momento da execução, em vez de depender apenas dos valores padrão. Adicione um tratamento de erro (`try/except`) para caso o microfone não seja detectado.

In [ ]:
import sounddevice as sd
import scipy.io.wavfile as wav
import time

def gravar_audio_v2(nome_arquivo="audio.wav", duracao=5, fs=44100, canais=1):
    print("Preparando para gravar...")
    for i in range(3, 0, -1):
        print(i)
        time.sleep(1)

    try:
        print("Gravando...")
        audio = sd.rec(int(duracao * fs), samplerate=fs, channels=canais)
        sd.wait()
        wav.write(nome_arquivo, fs, audio)
        print(f"Áudio '{nome_arquivo}' gravado com sucesso!")
    except sd.PortAudioError as e:
        print(f"Erro no dispositivo de áudio (Microfone não detectado?): {e}")
    except Exception as e:
        print(f"Ocorreu um erro inesperado: {e}")

# Exemplo de uso:
gravar_audio_v2(duracao=3, fs=48000)

### Exercício 2: Visualização de Dados Acústicos
Antes de enviar o áudio para a transcrição, carregue o arquivo `audio.wav` utilizando bibliotecas como `numpy` e `matplotlib` e plote o gráfico de forma de onda (waveform) do áudio gravado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
import os

def plotar_waveform(arquivo="audio.wav"):
    if not os.path.exists(arquivo):
        print("Arquivo de áudio não encontrado para plotagem.")
        return
        
    fs, data = wav.read(arquivo)
    
    # Se o áudio for estéreo, pegar apenas o primeiro canal para visualização
    if len(data.shape) > 1:
        data = data[:, 0]
        
    tempo = np.linspace(0., len(data) / fs, len(data))
    
    plt.figure(figsize=(10, 4))
    plt.plot(tempo, data, color="#1f77b4")
    plt.title("Forma de Onda do Áudio")
    plt.ylabel("Amplitude")
    plt.xlabel("Tempo [s]")
    plt.grid(True)
    plt.show()

# Exemplo de uso:
plotar_waveform()

### Exercício 3: Análise de Modelos Whisper
O código original utiliza o modelo `"base"` do Whisper. Altere o código para carregar os modelos `"tiny"` e `"small"`. Execute testes de fala e documente as diferenças observadas.

In [ ]:
import whisper
import time

def testar_modelos_whisper(arquivo="audio.wav"):
    modelos = ["tiny", "small"]
    
    for nome_modelo in modelos:
        print(f"\nCarregando modelo: {nome_modelo}...")
        inicio_carga = time.time()
        modelo = whisper.load_model(nome_modelo)
        print(f"Tempo de carga: {time.time() - inicio_carga:.2f}s")
        
        inicio_transcricao = time.time()
        resultado = modelo.transcribe(arquivo, language="pt")
        tempo_transcricao = time.time() - inicio_transcricao
        
        print(f"Texto: {resultado['text']}")
        print(f"Tempo de transcrição: {tempo_transcricao:.2f}s")

"""
DOCUMENTAÇÃO ESPERADA DO ALUNO:
- Modelo 'tiny': Carrega muito rápido e consome pouca memória RAM/VRAM. O tempo de transcrição é o menor de todos. Porém, a precisão para palavras complexas, sotaques ou ruído de fundo é consideravelmente menor.
- Modelo 'small': Demora mais para carregar e para processar a inferência. No entanto, o entendimento do contexto geral da frase é bem superior ao 'tiny' e 'base', cometendo menos erros ortográficos em português.
"""

### Exercício 4: Tratamento de Exceções em Arquivos
Atualize a função `transcrever_audio` para verificar se o arquivo `audio.wav` realmente existe no diretório antes de chamar o modelo.

In [ ]:
import os
import whisper

def transcrever_audio_seguro(arquivo="audio.wav"):
    if not os.path.exists(arquivo):
        return "Erro: O arquivo de áudio não foi encontrado. Grave o áudio primeiro."
        
    print("Transcrevendo...")
    try:
        modelo = whisper.load_model("base")
        resultado = modelo.transcribe(arquivo, language="pt")
        texto = resultado["text"]
        print("Texto reconhecido:", texto)
        return texto
    except Exception as e:
        return f"Erro durante a transcrição: {e}"

### Exercício 5: Evolução do Motor de Respostas (PLN)
Substitua a lógica `if/else` simples implementando a biblioteca `spaCy` para identificar a intenção da frase através de lematização e análise de tokens.

In [ ]:
# Requisito no terminal: python -m spacy download pt_core_news_sm
import spacy
import time

try:
    nlp = spacy.load("pt_core_news_sm")
except:
    print("Aviso: Modelo do spaCy não encontrado. Instale com: python -m spacy download pt_core_news_sm")
    nlp = None

def gerar_resposta_inteligente(texto):
    if not nlp:
        return "Erro: spaCy não inicializado."
        
    doc = nlp(texto.lower())
    lemmas = [token.lemma_ for token in doc]
    
    # Intenção de Saudação
    if any(palavra in lemmas for palavra in ["olá", "oi", "saudação", "e aí"]):
        return "Olá! Que bom falar com você. Como posso ajudar?"
        
    # Intenção de Identidade
    elif "nome" in lemmas or "quem" in lemmas:
        return "Eu sou o assistente virtual da nossa disciplina de NLP."
        
    # Intenção de Tempo
    elif "hora" in lemmas or "horário" in lemmas:
        return time.strftime("Neste exato momento, são %H horas e %M minutos.")
        
    else:
        return "Ainda estou aprendendo e não consegui identificar sua intenção."

# Exemplo de uso:
print(gerar_resposta_inteligente("Queria saber que horas são agora"))# Requisito no terminal: python -m spacy download pt_core_news_sm
import spacy
import time

try:
    nlp = spacy.load("pt_core_news_sm")
except:
    print("Aviso: Modelo do spaCy não encontrado. Instale com: python -m spacy download pt_core_news_sm")
    nlp = None

def gerar_resposta_inteligente(texto):
    if not nlp:
        return "Erro: spaCy não inicializado."
        
    doc = nlp(texto.lower())
    lemmas = [token.lemma_ for token in doc]
    
    # Intenção de Saudação
    if any(palavra in lemmas for palavra in ["olá", "oi", "saudação", "e aí"]):
        return "Olá! Que bom falar com você. Como posso ajudar?"
        
    # Intenção de Identidade
    elif "nome" in lemmas or "quem" in lemmas:
        return "Eu sou o assistente virtual da nossa disciplina de NLP."
        
    # Intenção de Tempo
    elif "hora" in lemmas or "horário" in lemmas:
        return time.strftime("Neste exato momento, são %H horas e %M minutos.")
        
    else:
        return "Ainda estou aprendendo e não consegui identificar sua intenção."

# Exemplo de uso:
print(gerar_resposta_inteligente("Queria saber que horas são agora"))

### Exercício 6: Assistente Contínuo
Refatore a função `assistente()` inserindo um laço `while True` para que funcione continuamente. Condição de parada: a palavra "desligar" ou "encerrar".

In [ ]:
def assistente_continuo():
    print("Iniciando assistente. Diga 'desligar' para encerrar.")
    
    while True:
        gravar_audio_v2("audio_temp.wav", duracao=4)
        texto = transcrever_audio_seguro("audio_temp.wav")
        
        if not texto or "Erro" in texto:
            continue
            
        texto_limpo = texto.lower()
        
        # Condição de saída
        if "desligar" in texto_limpo or "encerrar" in texto_limpo:
            print("Encerrando o assistente. Até logo!")
            sintetizar_voz("Desligando os sistemas. Até a próxima aula.")
            break
            
        resposta = gerar_resposta(texto) # Pode usar a inteligente do Ex 5
        print("Assistente:", resposta)
        sintetizar_voz(resposta)

### Exercício 7: Personalização Dinâmica da Voz
Modifique a função `sintetizar_voz` para receber `rate` e `volume` como argumentos, além de permitir escolher entre voz masculina ou feminina iterando sobre a propriedade `voices`.

In [ ]:
import pyttsx3

def sintetizar_voz_dinamica(texto, rate=180, volume=1.0, genero="feminina"):
    engine = pyttsx3.init()
    engine.setProperty('rate', rate)
    engine.setProperty('volume', volume)
    
    voices = engine.getProperty('voices')
    voz_selecionada = None
    
    for voice in voices:
        # Filtra vozes em português
        if "brazil" in voice.name.lower() or "portuguese" in voice.name.lower():
            # Tenta inferir o gênero pelo nome das vozes do Windows (Maria = Fem, Daniel/Thiago = Masc)
            if genero == "feminina" and "maria" in voice.name.lower():
                voz_selecionada = voice.id
                break
            elif genero == "masculina" and ("daniel" in voice.name.lower() or "thiago" in voice.name.lower()):
                voz_selecionada = voice.id
                break
            # Se não achar o nome específico, pega a primeira disponível em português como fallback
            if voz_selecionada is None:
                voz_selecionada = voice.id
                
    if voz_selecionada:
        engine.setProperty('voice', voz_selecionada)
        
    engine.say(texto)
    engine.runAndWait()

### Exercício 8: Análise de Dependências do SO
Pesquise e explique por que as bibliotecas `pywin32` e `comtypes` são necessárias para o funcionamento da síntese de voz em sistemas Windows.

**Resposta Teórica (Gabarito):**
No Windows, o motor de conversão de texto em fala (TTS) padrão utilizado pelo sistema operacional é o SAPI5 (Speech Application Programming Interface). O pacote `pyttsx3` atua como um "wrapper" de alto nível. Para que o código Python consiga se comunicar com a API SAPI5 nativa do Windows (que foi construída em C/C++ usando o modelo de componentes COM da Microsoft), as bibliotecas `pywin32` e `comtypes` são exigidas. Elas criam a ponte de comunicação que permite ao Python invocar, instanciar e enviar comandos diretamente para os objetos COM do Windows responsáveis por gerar a voz.